# Activity types in London

<a href="https://colab.research.google.com/github/openactive-contrib/openactive-admin-dashboard/blob/main/notebooks/02_london_activity_types.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Which activities are on offer in London, and what share of London's future opportunities
carries each one. Reads `opportunities` and `feeds` from
`openactive-monitor.openactive_analytics`. Each query prints how much it will scan before it
runs, and stops if that is over `MAX_GB`.

In [ ]:
import sys

from google.cloud import bigquery

PROJECT = "openactive-monitor"
DATASET = "openactive-monitor.openactive_analytics"
MAX_GB = 5  # refuse any single query that would scan more than this

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()  # opens a Google sign-in; queries then run as your account

client = bigquery.Client(project=PROJECT)


def run_query(sql, params=None, max_gb=MAX_GB):
    """Dry-run, report the scan size, then run with a hard cap on bytes billed."""
    config = bigquery.QueryJobConfig(query_parameters=params or [])
    dry = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=params or [], dry_run=True, use_query_cache=False
        ),
    )
    gb = dry.total_bytes_processed / 1e9
    print(f"This query will scan {gb:.2f} GB.")
    if gb > max_gb:
        raise RuntimeError(f"Scan of {gb:.2f} GB exceeds max_gb={max_gb}; narrow the query.")
    config.maximum_bytes_billed = int(max_gb * 1e9)
    return client.query(sql, job_config=config).to_dataframe()

In [ ]:
REGION = "London"  # @param {type:"string"}
TOP_N = 25  # @param {type:"integer"}

## What is counted

An *opportunity* here is one row of `opportunities` that:

- comes from a **regular** feed (preview feeds are excluded),
- is an **activity kind**, so not `FacilityUse`, `IndividualFacilityUse` or `Slot`, which
  describe facilities rather than activities,
- **starts from today** (`startDate` from midnight UTC today onwards).

Activity labels are the publishers' own text, lower-cased and trimmed, and counted once per
opportunity. A session inherits its series' activity, so sessions are labelled even when
the label is only on the series.

In [ ]:
from google.cloud.bigquery import ScalarQueryParameter

BASE_SQL = f"""
WITH latest_feeds AS (
  SELECT id, is_regular
  FROM `{DATASET}.feeds`
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY last_access DESC) = 1
),
future_activity_opportunities AS (
  SELECT
    o.region_name,
    ARRAY(
      SELECT DISTINCT LOWER(TRIM(label))
      FROM UNNEST(JSON_VALUE_ARRAY(o.activity)) AS label
      WHERE TRIM(label) != ''
    ) AS activities
  FROM `{DATASET}.opportunities` AS o
  JOIN latest_feeds AS f
    ON o.feed_id = f.id
  WHERE f.is_regular
    AND NOT COALESCE(o.kind IN ('FacilityUse', 'IndividualFacilityUse', 'Slot'), FALSE)
    AND o.startDate >= TIMESTAMP(CURRENT_DATE())
)
"""

## Opportunities by region

The same opportunities across every region, to show how many have no region (their location
could not be resolved) and to check that the region parameter matches a real value.

In [ ]:
sql_regions = (
    BASE_SQL
    + """
SELECT
  COALESCE(region_name, '(no region)') AS region,
  COUNT(*) AS opportunities,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_all
FROM future_activity_opportunities
GROUP BY region
ORDER BY opportunities DESC
"""
)
regions = run_query(sql_regions)
display(regions)

if REGION not in set(regions["region"]):
    raise ValueError(f"No opportunities have region_name = {REGION!r}; pick one listed above.")

## Activity types in the region

For each activity label, the number of the region's opportunities that carry it, as a
percentage of all the region's opportunities. An opportunity can carry several labels, so
the percentages add up to more than 100%.

In [ ]:
sql_activities = (
    BASE_SQL
    + """,
in_region AS (
  SELECT activities
  FROM future_activity_opportunities
  WHERE region_name = @region
),
totals AS (
  SELECT
    COUNT(*) AS n_opportunities,
    COUNTIF(ARRAY_LENGTH(activities) = 0) AS n_without_activity
  FROM in_region
)
SELECT
  activity,
  COUNT(*) AS opportunities,
  ROUND(100 * COUNT(*) / ANY_VALUE(totals.n_opportunities), 2) AS pct_of_opportunities,
  ANY_VALUE(totals.n_opportunities) AS region_opportunities,
  ANY_VALUE(totals.n_without_activity) AS region_without_activity
FROM in_region, UNNEST(activities) AS activity
CROSS JOIN totals
GROUP BY activity
ORDER BY opportunities DESC, activity
"""
)
activities = run_query(sql_activities, [ScalarQueryParameter("region", "STRING", REGION)])

total = int(activities["region_opportunities"].iloc[0]) if len(activities) else 0
unlabelled = int(activities["region_without_activity"].iloc[0]) if len(activities) else 0
print(f"{total:,} future opportunities in {REGION}.")
if total:
    share = 100 * unlabelled / total
    print(f"{unlabelled:,} ({share:.1f}%) have no activity label and appear in no row below.")
print(f"{len(activities):,} distinct activity labels.")

activity_table = activities[["activity", "opportunities", "pct_of_opportunities"]]
display(activity_table)

In [ ]:
import plotly.express as px

top = activity_table.head(TOP_N)
fig = px.bar(
    top,
    x="pct_of_opportunities",
    y="activity",
    orientation="h",
    title=f"Top {len(top)} activity types in {REGION}, % of future opportunities",
    labels={"pct_of_opportunities": "% of future opportunities", "activity": ""},
    height=max(400, 22 * len(top)),
)
fig.update_yaxes(autorange="reversed")
fig.show()

## Notes

- The data is the current state of the daily batch, not live.
- Regular feeds only. The pre-computed `active_opportunities_summary` table gives a similar
  breakdown but includes preview feeds, so its figures will differ.
- Labels are free text: `yoga` and `hatha yoga` are separate rows. No mapping to the
  OpenActive activity list is applied.
- Opportunities whose location could not be resolved have no region and are not counted in
  any region; the first table shows how many.
- A series whose own start date is in the future is counted alongside its sessions; sessions
  embedded inside a series (rather than published as rows) are not counted.